<a href="https://colab.research.google.com/github/BenayaL/Cloud_Computing_Project/blob/main/cloudPrac2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# Loading the data
file_path = '/content/drive/My Drive/Studies/Cloud/students.json'

try:
    df = pd.read_json(file_path)
except FileNotFoundError:
    print(f"Error: Could not find the file at {file_path}")
    df = pd.DataFrame([{"firstName": "Error", "lastName": "Missing File", "email": "none", "coursesTaken": [], "interestingLink": ""}])

df['Full Name'] = df['firstName'] + " " + df['lastName']
if 'favoriteProgram' not in df.columns:
    df['favoriteProgram'] = ""

# Creating the UI Widgets
title = widgets.HTML("<h2>🎓 Student Information Editor</h2>")

student_dropdown = widgets.Dropdown(
    options=['-- Select a Student --'] + df['Full Name'].tolist(),
    description='Student:',
    disabled=False,
    layout=widgets.Layout(width='400px', margin='0 0 15px 0') # Added some spacing
)

info_html = widgets.HTML(value="")
status_output = widgets.Output()

program_input = widgets.Text(description='Fav Program:', disabled=True, layout=widgets.Layout(width='300px'))
save_button = widgets.Button(description='Save to File', button_style='success', disabled=True, icon='save')

def on_dropdown_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        selected_name = change['new']
        status_output.clear_output()

        if selected_name == '-- Select a Student --':
            info_html.value = ""
            program_input.value = ''
            program_input.disabled = True
            save_button.disabled = True
            return

        program_input.disabled = False
        save_button.disabled = False

        # Grab the student's data
        student_data = df[df['Full Name'] == selected_name].iloc[0]
        courses_list = student_data['coursesTaken']
        formatted_courses = ", ".join(courses_list) if isinstance(courses_list, list) else courses_list
        link = student_data['interestingLink']

        clickable_link = f'<a href="{link}" target="_blank" style="color: #021d4f; text-decoration: none;"><b>Open Link ↗</b></a>' if link and link != "N/A" else "No link provided"

        # Constructing the styled HTML table
        html_content = f"""
        <style>
            .student-table {{
                width: 100%;
                max-width: 600px;
                border-collapse: collapse;
                font-family: Arial, sans-serif;
                margin-bottom: 20px;
                box-shadow: 0 2px 5px rgba(0,0,0,0.1);
            }}
            .student-table th, .student-table td {{
                border: 1px solid #e0e0e0;
                padding: 12px;
                text-align: left;
            }}
            .student-table th {{
                background-color: #2C3E50;
                color: white;
                width: 130px;
                font-weight: bold;
            }}
            .student-table tr:nth-child(even) {{
                background-color: DimGray;
            }}
            .student-table tr:hover {{
                background-color: #6390eb;
            }}
        </style>

        <table class="student-table">
            <tr><th>Name</th><td>{selected_name}</td></tr>
            <tr><th>Email</th><td>{student_data['email']}</td></tr>
            <tr><th>Courses</th><td>{formatted_courses}</td></tr>
            <tr><th>Link</th><td>{clickable_link}</td></tr>
        </table>
        """
        # Injecting the HTML into the widget
        info_html.value = html_content

        current_fav = student_data['favoriteProgram']
        program_input.value = current_fav if pd.notna(current_fav) else ""

def on_save_clicked(b):
    selected_name = student_dropdown.value
    new_program = program_input.value

    row_index = df[df['Full Name'] == selected_name].index[0]
    df.at[row_index, 'favoriteProgram'] = new_program

    df_to_save = df.drop(columns=['Full Name'])
    df_to_save.to_json(file_path, orient='records', indent=2)

    with status_output:
        clear_output()
        print(f"✅ Successfully saved '{new_program}' to {file_path} for {selected_name}!")

student_dropdown.observe(on_dropdown_change)
save_button.on_click(on_save_clicked)

edit_row = widgets.HBox([program_input, save_button], layout=widgets.Layout(margin='10px 0'))
ui_layout = widgets.VBox([
    title,
    student_dropdown,
    info_html,
    edit_row,
    status_output
], layout=widgets.Layout(padding='20px', border='1px solid #ddd', border_radius='10px', width='650px'))

display(ui_layout)